# Aula 3 · Lab 1 — Tópicos, partições e réplicas

**Objetivo:** criar tópicos, entender como partições e réplicas se distribuem entre os 3 brokers
do nosso cluster e ler esses metadados por código e pela linha de comando.

Ambiente: `kafka1`, `kafka2` e `kafka3` (modo KRaft, sem ZooKeeper), acessíveis em `kafka1:9092,kafka2:9092,kafka3:9092`.
Abra também a **Kafka UI** (porta 8080) numa aba ao lado: tudo o que fizermos aqui aparece lá.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic, ConfigResource
from comum.config import BOOTSTRAP

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
meta = admin.list_topics(timeout=10)

print("Cluster id:", meta.cluster_id)
print("Controller ativo:", meta.controller_id)
for b in sorted(meta.brokers.values(), key=lambda b: b.id):
    print(f"  broker {b.id} -> {b.host}:{b.port}")

## 1. Criando um tópico

Um tópico é dividido em **partições** (unidade de paralelismo e de ordenação) e cada partição é
copiada em **réplicas** (unidade de tolerância a falhas). Vamos criar `aula3-pedidos` com
6 partições e fator de replicação 3.

In [ ]:
TOPICO = "aula3-pedidos"

futuros = admin.create_topics([NewTopic(TOPICO, num_partitions=6, replication_factor=3,
                                        config={"retention.ms": str(24 * 3600 * 1000)})])
for nome, f in futuros.items():
    try:
        f.result()
        print("criado:", nome)
    except Exception as e:
        print("aviso:", e)

## 2. Quem é líder de cada partição?

Cada partição tem **um líder** (recebe escritas e leituras) e seguidores. O conjunto **ISR**
(*in-sync replicas*) são as réplicas que estão em dia com o líder.

In [ ]:
def descrever(topico):
    t = admin.list_topics(topic=topico, timeout=10).topics[topico]
    print(f"{topico}: {len(t.partitions)} partições")
    for p in sorted(t.partitions.values(), key=lambda p: p.id):
        print(f"  partição {p.id}: líder={p.leader} réplicas={p.replicas} ISR={p.isrs}")

descrever(TOPICO)

Repare que os líderes ficam espalhados entre os brokers 1, 2 e 3: assim a carga de escrita se divide.

### O mesmo pela linha de comando
Abra um **terminal** no Jupyter (File → New → Terminal) e rode:

```bash
kafka-topics.sh --bootstrap-server kafka1:9092 --list
kafka-topics.sh --bootstrap-server kafka1:9092 --describe --topic aula3-pedidos
kafka-configs.sh --bootstrap-server kafka1:9092 --entity-type topics --entity-name aula3-pedidos --describe
```

## 3. Configurações de um tópico
Configurações importantes: `retention.ms` (quanto tempo guardar), `cleanup.policy`
(`delete` ou `compact`) e `min.insync.replicas` (usado com `acks=all`, veremos no Lab 4).

In [ ]:
recurso = ConfigResource(ConfigResource.Type.TOPIC, TOPICO)
configs = admin.describe_configs([recurso])[recurso].result()
for chave in ["retention.ms", "cleanup.policy", "min.insync.replicas", "segment.bytes"]:
    print(f"{chave:22s} = {configs[chave].value}")

## 4. Tópicos do curso
Os labs seguintes usam os tópicos do gerador de eventos da loja (`cliques`, `pedidos`,
`pagamentos`, `logins`) e um de saída (`alertas`). Vamos criá-los agora.

In [ ]:
from comum.config import criar_topicos
criar_topicos()

## Exercícios
1. Crie um tópico `aula3-teste` com **1 partição** e replicação 3. Pela Kafka UI, descubra quem é o líder.
2. Tente criar um tópico com `replication_factor=4`. Que erro aparece? Por quê?
3. Aumente o número de partições de `aula3-teste` para 4 (`kafka-topics.sh --alter --partitions 4`).
   Dá para **diminuir** depois? Que problema isso causaria para mensagens com chave?
4. Crie um tópico compactado (`cleanup.policy=compact`). Em que caso de uso ele faz sentido?